# AI 드론 데이터 코딩 실습 노트 (완성본)
### Ⅱ. 드론 데이터 처리 및 분석

이 노트는 실습용 노트의 빈칸을 **모두 채운 완성본**입니다. 셀마다 실행 결과가 함께 들어 있습니다.

- 수업을 마친 뒤 자신의 노트와 견주어 보거나, 막힌 곳을 확인할 때 씁니다.
- `해 보기`는 정답이 하나가 아니므로 비워 두었습니다.
- 가상 드론의 바람과 센서 값에는 흔들림이 들어 있어, 다시 실행하면 소수점 아래가 조금 달라질 수 있는 예제가 있습니다.

| 소단원 | 예제 |
|---|---|
| 2-1 표준·외부 라이브러리 활용 | 2-1, 2-2, 2-3, 2-4 |
| 2-2 데이터프레임 활용 | 2-5, 2-6, 2-7, 2-8, 2-9, 2-10 |
| 2-3 데이터 수집과 전처리 | 2-11, 2-12, 2-13, 2-14, 2-15 |
| 2-4 기상 및 비행 데이터 분석 | 2-16, 2-17, 2-18 |

## 0. 준비하기

아래 셀은 **실습 준비용**입니다. 내용을 이해하지 않아도 괜찮습니다. 실행만 하면 다음 일을 대신 해 줍니다.

- 가상 드론 `drone_sim.py`를 인터넷에서 내려받습니다. 내려받지 못했을 때만 **파일 선택 창**이 뜹니다. 그때는 선생님이 나누어 준 파일을 골라 올립니다.
- 그래프에 한글이 깨지지 않게 해 주는 `font_setup.py`를 만들어 둡니다.

> 코랩은 연결이 끊기면 파일을 잊습니다. 다음 시간에 다시 열었다면 이 셀부터 다시 실행합니다.

In [2]:
#@title ▶ 준비 셀 (실행만 하세요)
import os, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# 1) 한글 글꼴 설정 모듈(font_setup.py)을 만들어 둔다 (교재 부록 2와 같은 내용)
FONT_SETUP = '''"""font_setup.py - 그래프에 한글이 깨지지 않도록 글꼴을 정해 주는 모듈
그래프를 그리는 예제에서 import font_setup 한 줄만 적으면 된다.
코랩에는 한글 글꼴이 없으므로, 처음 한 번 나눔고딕을 설치한다.
"""
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 1) 코랩이면 나눔고딕을 설치해서 등록한다
if os.path.exists("/content") and not os.path.exists(NANUM):
    os.system("apt-get -qq install fonts-nanum > /dev/null")
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)

# 2) 운영체제마다 들어 있는 한글 글꼴이 다르므로, 있는 것을 찾아 쓴다
candidates = ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR"]
installed = {f.name for f in fm.fontManager.ttflist}
for name in candidates:
    if name in installed:
        plt.rcParams["font.family"] = name
        break
else:
    print("한글 글꼴을 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

plt.rcParams["axes.unicode_minus"] = False   # 음수 기호가 깨지지 않도록
'''
if not os.path.exists("font_setup.py"):
    with open("font_setup.py", "w", encoding="utf-8") as f:
        f.write(FONT_SETUP)

# 2) 가상 드론(drone_sim.py)을 내려받는다. 받지 못하면 파일 선택 창을 띄운다
import urllib.request
URL = "https://raw.githubusercontent.com/ralralra/dronebook/main/data/drone_sim.py"
try:
    urllib.request.urlretrieve(URL, "drone_sim.py")
except Exception:
    print("drone_sim.py 를 내려받지 못했습니다.")
if not os.path.exists("drone_sim.py") and IN_COLAB:
    from google.colab import files
    print("선생님이 나누어 준 drone_sim.py 를 골라 올려 주세요.")
    files.upload()

for name in ["drone_sim.py", "font_setup.py"]:
    print(("✓ 준비됨  " if os.path.exists(name) else "✗ 없음    ") + name)


✓ 준비됨  drone_sim.py
✓ 준비됨  font_setup.py


# 2-1 표준·외부 라이브러리 활용

**학습 목표**

- 라이브러리의 개념과 표준·외부 라이브러리의 차이를 설명할 수 있습니다.
- import로 라이브러리를 불러와 필요한 기능을 쓸 수 있습니다.
- 공식 문서에서 함수의 사용 방법을 찾아 적용할 수 있습니다.

<sub>관련 성취기준 [드코1 02-01-01] / [드코1 02-01-02]</sub>

> **생각 열기**  
> 두 지점 사이의 거리를 구하려면 제곱근이 필요합니다. 그렇다고 제곱근 구하는 방법을 직접 만들어야 할까요? 다행히 이미 누군가 만들어 둔 것을 꺼내 쓸 수 있습니다. 사실 우리는 이미 그렇게 해 왔습니다. from drone\_sim import \*도 누군가 만들어 둔 드론 기능을 꺼내 쓰는 문장이었습니다.

<sub>교재에서 읽기: ■ 1. 라이브러리란</sub>

## ■ 2. 라이브러리 불러오기

라이브러리는 import로 불러옵니다. 불러오는 방법은 세 가지입니다.

**예제 2-1. 세 가지 불러오기 방법**

In [3]:
# 예제 2-1. 세 가지 불러오기 방법
import math                 # 방법 1: 통째로
print(math.sqrt(16))

from math import sqrt       # 방법 2: 필요한 것만
print(sqrt(25))

import math as m            # 방법 3: 짧은 이름으로
print(m.pi)

4.0
5.0
3.141592653589793


> **한 걸음 더 · 어떤 방법을 쓸까요**  
> 방법 1은 math.이 붙어 어느 라이브러리의 기능인지 분명히 드러나므로 가장 안전합니다. 방법 2는 짧게 쓸 수 있지만 이름이 겹칠 위험이 있습니다. 방법 3은 import pandas as pd처럼 많은 사람이 쓰는 약속된 줄임말이 있을 때 주로 씁니다. 우리가 쓴 from drone\_sim import \*의 \*는 '전부'라는 뜻으로, 방법 2의 가장 넓은 형태입니다. 편하지만 어떤 이름이 들어왔는지 한눈에 보이지 않으므로, 드론 라이브러리처럼 안내서가 그렇게 쓰라고 할 때만 씁니다.

## ■ 3. math: 거리 계산에 쓰기

두 지점의 좌표를 알 때 직선거리는 피타고라스 정리로 구합니다. math.sqrt()는 제곱근을, math.dist()는 두 점 사이의 거리를 바로 구해 줍니다.

**예제 2-2. 경유지 사이의 거리 구하기**

In [4]:
# 예제 2-2. 경유지 사이의 거리 구하기
import math

start = (0, 0)          # 출발지 좌표(m)
target = (300, 400)     # 배송지 좌표(m)

dx = target[0] - start[0]
dy = target[1] - start[1]
distance = math.sqrt(dx ** 2 + dy ** 2)

print("직접 계산:", distance, "m")
print("math.dist:", math.dist(start, target), "m")

직접 계산: 500.0 m
math.dist: 500.0 m


## ■ 4. time: 걸린 시간 재기

time 라이브러리는 시간을 다룹니다. time.sleep(초)는 프로그램을 잠시 멈추고, time.time()은 현재 시각을 숫자로 알려 줍니다. 두 시각의 차이로 걸린 시간을 잴 수 있습니다.

**예제 2-3. 작업에 걸린 시간 재기**

In [5]:
# 예제 2-3. 작업에 걸린 시간 재기
import time

start = time.time()              # 시작 시각 기록
total = 0
for i in range(1, 100001):       # 1부터 10만까지 더하기
    total = total + i
end = time.time()                # 끝난 시각 기록

print("합계:", total)
print("걸린 시간이 0.5초보다 짧은가?", (end - start) < 0.5)

합계: 5000050000
걸린 시간이 0.5초보다 짧은가? True


> **짚고 가기 · 실행할 때마다 달라지는 값**  
> time.time()으로 잴 시간은 컴퓨터의 상태에 따라 실행할 때마다 조금씩 다릅니다. 코랩은 구글 서버를 여러 사람이 나눠 쓰므로 더욱 그렇습니다. 이런 값은 그대로 비교하지 않고, 위 예제처럼 '기준보다 짧은가'로 판단하거나 여러 번 재어 평균을 냅니다.

## ■ 5. csv: 비행 기록 파일 읽기

Ⅰ단원에서 sim\_save\_log()로 저장한 flight\_log.csv 파일을 열어 봅시다. CSV는 값을 쉼표로 구분해 저장하는 가장 단순한 표 형식 파일입니다. 코랩에서 저장한 파일은 왼쪽 파일 창에 나타납니다.

**예제 2-4. 비행 기록 파일 읽어 보기**

In [6]:
# 예제 2-4. 비행 기록 파일 읽어 보기
from drone_sim import *
import csv

# 먼저 짧은 비행으로 기록 파일을 만든다
drone = Drone()
drone.pair()
drone.takeoff()
drone.move_forward(200)
drone.land()
drone.sim_save_log()          # 가상 전용
drone.close()

# 저장된 파일을 읽는다
with open("flight_log.csv", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["시간"], row["동작"], row["배터리"])

[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 200cm
[가상] 착륙
[가상] 비행 기록 6줄을 flight_log.csv 에 저장했습니다.
[가상] 연결을 닫았습니다.
0.0 준비 100.0
0.0 연결 100.0
2.0 이륙 99.4
3.0 정지비행 99.2
7.0 move_forward 98.2
9.0 착륙 97.7


with open(...)은 파일을 열고 작업이 끝나면 자동으로 닫아 주는 안전한 방법입니다. encoding="utf-8-sig"는 한글이 깨지지 않게 하는 설정입니다. 기록 파일의 배터리는 99.4처럼 소수점까지 남습니다. 화면에 보여 주는 get\_battery()는 정수였지만, 기록에는 가상 드론 안의 값을 그대로 적었기 때문입니다. 직접 나오는 값과 기록된 값이 다를 수 있다는 것은 데이터를 다룰 때 항상 확인할 점입니다.

## ■ 6. 외부 라이브러리 설치하기

다음 소단원부터 쓸 pandas와 matplotlib는 외부 라이브러리입니다. 코랩에는 이 두 라이브러리가 미리 설치되어 있어 바로 import하면 됩니다. 설치되어 있지 않은 라이브러리는 코드 셀에 느낌표(!)를 붙여 설치합니다. 파이참에서는 1-5에서 codrone-edu를 설치한 것처럼 Settings의 Python Interpreter 메뉴를 씁니다.

```python
!pip install codrone-edu        # 코랩: ! 는 '파이썬이 아니라 명령어로 실행'이라는 표시

import pandas as pd
print(pd.__version__)          # 설치된 버전 확인
```

코랩에도 드론 라이브러리는 설치됩니다. 다만 1-1에서 살펴본 것처럼 조종기를 볼 수 없어서 drone.pair()에서 멈춥니다. 설치되는 것과 쓸 수 있는 것은 다릅니다. 코랩에서 설치한 라이브러리는 올린 파일처럼 런타임이 끊기면 사라집니다. 노트 맨 위에 설치 셀을 모아 두면 다음 시간에 한 번에 다시 실행할 수 있습니다.

> **안전 수칙 · 설치할 때 지킬 것**  
> 라이브러리는 공식 배포처에서만 설치합니다. 인터넷에서 찾은 출처가 불분명한 파일을 내려받아 설치하면 악성 코드가 함께 들어올 수 있습니다. 이름이 비슷한 가짜 라이브러리도 있으므로 이름의 철자를 정확히 확인합니다.

## ■ 7. 공식 문서 찾아 읽기

라이브러리의 모든 기능을 외울 수는 없습니다. 중요한 것은 필요할 때 찾아 쓰는 능력입니다. 공식 문서에는 함수마다 다음 세 가지가 적혀 있으니 이 순서로 확인합니다.

1. 무엇을 하는 함수인가
2. 어떤 값을 넣어야 하는가(매개변수)
3. 무엇을 돌려주는가(반환값)

1-5에서 move\_forward()의 인자를 표로 정리한 것(표 1-10)이 바로 이 순서입니다. 코랩에서는 help(math.dist)를 실행하거나 함수 이름 뒤에 물음표를 붙여(math.dist?) 설명을 바로 볼 수도 있습니다.

> **AI와 함께 · 검색 결과를 그대로 믿지 않기**  
> 인터넷 검색이나 생성형 AI가 알려 준 함수 사용법이 실제와 다르거나 옛 버전인 경우가 있습니다. 이름은 그럴듯한데 존재하지 않는 함수를 알려 주기도 합니다. 반드시 공식 문서에서 확인하고, 짧은 코드로 직접 실행해 본 뒤 씁니다. 이 습관은 Ⅳ단원에서 본격적으로 다룹니다.

### ====== 해 보기 ======

**라이브러리 조사하기**

모둠별로 표준 라이브러리 하나를 정해 조사하고 소개해 봅시다.

- random(난수), datetime(날짜·시각), statistics(평균·중앙값) 가운데서 고릅니다.
- 그 라이브러리의 기능 가운데 드론과 관련지어 쓸 수 있는 것 두 가지를 찾습니다.
- 짧은 예제 코드를 만들어 실행해 보고, 공식 문서의 어느 부분을 참고했는지 함께 밝힙니다.

In [7]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 라이브러리는 미리 만들어 둔 기능의 꾸러미로, 표준과 외부 라이브러리로 나뉩니다.
- import로 불러오며, as를 써서 짧은 이름을 붙일 수 있습니다.
- math로 거리를, time으로 시간을, csv로 표 형식 파일을 다룹니다.
- 외부 라이브러리는 공식 배포처에서 설치하고, 사용법은 공식 문서에서 확인합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 2-2 데이터프레임 활용

**학습 목표**

- 데이터프레임의 구조를 이해하고 CSV 파일을 불러올 수 있습니다.
- 필요한 행과 열을 고르고 조건에 맞는 데이터를 걸러낼 수 있습니다.
- 정렬과 집계로 비행 데이터의 특징을 확인할 수 있습니다.

<sub>관련 성취기준 [드코1 02-02-01] / [드코1 02-02-02]</sub>

> **생각 열기**  
> 비행 기록이 다섯 줄이면 눈으로 읽을 수 있습니다. 그런데 1초에 한 번씩 7분 동안 기록하면 420줄이 됩니다. 이 많은 줄에서 '배터리를 가장 많이 쓴 구간'을 어떻게 찾을까요?

<sub>교재에서 읽기: ■ 1. 데이터프레임이란</sub>

## ■ 2. CSV 파일 불러오기

pandas는 보통 pd라는 줄임말로 불러옵니다. read\_csv()로 파일을 읽으면 데이터프레임이 만들어집니다.

**예제 2-5. 비행 기록 불러오기**

In [8]:
# 예제 2-5. 비행 기록 불러오기
from drone_sim import *
import pandas as pd

# 분석할 비행 기록 만들기
drone = Drone()
drone.pair()
drone.takeoff()
for d in [100, 250, 80]:
    drone.move_forward(d)
    drone.turn_right(90)
drone.land()
drone.sim_save_log()        # 가상 전용
drone.close()

df = pd.read_csv("flight_log.csv")
print(df)

[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 100cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 250cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 80cm
[가상] 오른쪽으로 90도 회전
[가상] 착륙
[가상] 비행 기록 11줄을 flight_log.csv 에 저장했습니다.
[가상] 연결을 닫았습니다.
      시간            동작      x      y     z     방향    배터리   풍속
0    0.0            준비    0.0    0.0   0.0    0.0  100.0  0.0
1    0.0            연결    0.0    0.0   0.0    0.0  100.0  0.0
2    2.0            이륙    0.0    0.0  80.0    0.0   99.4  0.0
3    3.0          정지비행    0.0    0.0  80.0    0.0   99.2  0.0
4    5.0  move_forward  100.0    0.0  80.0    0.0   98.7  0.0
5    6.0    turn_right  100.0    0.0  80.0  -90.0   98.4  0.0
6   11.0  move_forward  100.0 -250.0  80.0  -90.0   97.2  0.0
7   12.0    turn_right  100.0 -250.0  80.0 -180.0   96.9  0.0
8   13.6  move_forward   20.0 -250.0  80.0 -180.0   96.5  0.0
9   14.6    turn_right   20.0 -250.0  80.0   90.0   96.3  0.0
10  16.6            착륙   20.0 -250.0   0.0   90.0   95.8  0.0


열을 하나씩 읽어 봅시다. x는 앞뒤, y는 왼쪽(+)과 오른쪽(−), z는 높이입니다. 오른쪽으로 돈 뒤 250 cm를 갔더니 y가 −250이 되었습니다. 방향도 왼쪽이 +이므로 오른쪽으로 돌 때마다 −90, −180으로 줄어듭니다. 세 번째에는 −270 대신 90이 나왔는데, 오른쪽으로 270도 돈 것과 왼쪽으로 90도 돈 것은 같은 방향이기 때문입니다.

> **짚고 가기 · 다음 시간에 이어서 할 때**  
> 예제 2-6부터는 예제 2-5가 만든 flight\_log.csv를 읽습니다. 코랩은 연결이 끊기면 올린 파일뿐 아니라 프로그램이 만든 파일도 함께 지웁니다. 그래서 다음 시간에 노트를 다시 열어 예제 2-6부터 실행하면 FileNotFoundError가 납니다. '그런 파일을 찾지 못했다'는 뜻입니다.  
> 이럴 때는 파일을 만드는 예제를 먼저 다시 실행합니다. 이어서 할 셀을 누른 뒤 메뉴에서 런타임 → 이전 셀 실행을 고르면, 그 위에 있는 셀을 처음부터 차례로 다시 실행해 줍니다.
> - flight\_log.csv: 예제 2-5가 만들고, 예제 2-6\~2-10이 읽습니다.
> - weather.csv: 예제 2-11이 만들고, 예제 2-12\~2-15가 읽습니다.
> - sim\_result.csv: 예제 3-4가 만들고, 예제 3-5와 3-7이 읽습니다.
> - flight\_history.csv: 예제 4-6이 만들고, 예제 4-7\~4-9가 읽습니다.


데이터가 많을 때는 전부 출력하지 않고 일부만 살펴봅니다.

**예제 2-6. 데이터 살펴보기**

In [9]:
# 예제 2-6. 데이터 살펴보기
import pandas as pd

df = pd.read_csv("flight_log.csv")
print("행과 열의 수:", df.shape)
print("열 이름:", list(df.columns))
print()
print("--- 처음 3줄 ---")
print(df.head(3))

행과 열의 수: (11, 8)
열 이름: ['시간', '동작', 'x', 'y', 'z', '방향', '배터리', '풍속']

--- 처음 3줄 ---
    시간  동작    x    y     z   방향    배터리   풍속
0  0.0  준비  0.0  0.0   0.0  0.0  100.0  0.0
1  0.0  연결  0.0  0.0   0.0  0.0  100.0  0.0
2  2.0  이륙  0.0  0.0  80.0  0.0   99.4  0.0


| 명령 | 하는 일 |
|---|---|
| df.shape | (행 수, 열 수)를 알려 줍니다 |
| df.columns | 열 이름의 목록을 알려 줍니다 |
| df.head(n) | 앞에서 n줄만 보여 줍니다. 숫자를 빼면 5줄입니다 |
| df.tail(n) | 뒤에서 n줄만 보여 줍니다 |
| df.info() | 열마다 자료형과 빈칸이 아닌 값의 수를 알려 줍니다 |

<sub>표 2-3. 데이터를 살펴보는 명령</sub>

> **한 걸음 더 · 코랩에서는 print를 빼 보세요**  
> 코랩에서 셀의 마지막 줄에 print(df) 대신 df만 쓰면, 글자 대신 줄과 칸이 그려진 표로 보여 줍니다. 한글 열 이름도 가지런히 정렬됩니다.

## ■ 3. 열과 행 고르기

열은 대괄호 안에 열 이름을 적어 꺼냅니다. 여러 열을 꺼낼 때는 열 이름의 리스트를 넣으므로 대괄호가 두 번 겉칩니다.

**예제 2-7. 필요한 열만 꺼내기**

In [10]:
# 예제 2-7. 필요한 열만 꺼내기
import pandas as pd

df = pd.read_csv("flight_log.csv")
print("--- 배터리 열만 ---")
print(df["배터리"].head(3))
print()
print("--- 시간과 배터리 두 열 ---")
print(df[["시간", "배터리"]].head(3))

--- 배터리 열만 ---
0    100.0
1    100.0
2     99.4
Name: 배터리, dtype: float64

--- 시간과 배터리 두 열 ---
    시간    배터리
0  0.0  100.0
1  0.0  100.0
2  2.0   99.4


## ■ 4. 조건으로 걸러내기

대괄호 안에 조건을 넣으면 조건을 만족하는 행만 골라낼 수 있습니다. 이를 필터링이라고 합니다.

**예제 2-8. 앞으로 간 기록만 골라내기**

In [11]:
# 예제 2-8. 앞으로 간 기록만 골라내기
import pandas as pd

df = pd.read_csv("flight_log.csv")
forward = df[df["동작"] == "move_forward"]
print(forward[["시간", "동작", "배터리"]])

low = df[df["배터리"] < 98]
print("\n배터리가 98 % 미만인 기록 수:", len(low))

     시간            동작   배터리
4   5.0  move_forward  98.7
6  11.0  move_forward  97.2
8  13.6  move_forward  96.5

배터리가 98 % 미만인 기록 수: 5


df["동작"] == "move\_forward"는 줄마다 True인지 False인지를 담은 목록을 만듭니다. df[...]는 그 가운데 True인 줄만 남깁니다. 1-3의 비교 연산자를 표 전체에 한 번에 적용한 셈입니다.

> **한 걸음 더 · 조건을 두 개 이상 쓸 때**  
> 파이썬의 and, or 대신 &amp;(그리고), |(또는)를 쓰고, 조건마다 반드시 소괄호로 묶습니다.

```python
df[(df["동작"] == "move_forward") & (df["배터리"] > 97)]
```

## ■ 5. 정렬하기

sort\_values()로 특정 열을 기준으로 줄을 세울 수 있습니다. ascending=False를 넣으면 큰 값부터 내림차순으로 정렬됩니다.

**예제 2-9. 배터리가 적게 남은 순서로 정렬**

In [12]:
# 예제 2-9. 배터리가 적게 남은 순서로 정렬
import pandas as pd

df = pd.read_csv("flight_log.csv")
sorted_df = df.sort_values("배터리")
print(sorted_df[["시간", "동작", "배터리"]].head(3))

      시간            동작   배터리
10  16.6            착륙  95.8
9   14.6    turn_right  96.3
8   13.6  move_forward  96.5


인덱스(10, 9, 8)는 정렬해도 원래 번호를 그대로 가지고 다닙니다. 그래서 정렬한 뒤에도 원래 몇 번째 기록이었는지 알 수 있습니다.

## ■ 6. 집계하기

여러 값을 하나의 수로 요약하는 것을 집계라고 합니다. 평균, 최댓값, 최솟값, 합계 등을 구할 수 있습니다.

**예제 2-10. 비행 데이터 요약하기**

In [13]:
# 예제 2-10. 비행 데이터 요약하기
import pandas as pd

df = pd.read_csv("flight_log.csv")
print("총 비행 시간 :", df["시간"].max(), "초")
print("최고 높이    :", df["z"].max(), "cm")
print("처음 배터리  :", df["배터리"].max(), "%")
print("마지막 배터리:", df["배터리"].min(), "%")
print("사용한 배터리:", round(df["배터리"].max() - df["배터리"].min(), 1), "%")
print("동작별 횟수  :")
print(df["동작"].value_counts())

총 비행 시간 : 16.6 초
최고 높이    : 80.0 cm
처음 배터리  : 100.0 %
마지막 배터리: 95.8 %
사용한 배터리: 4.2 %
동작별 횟수  :
동작
move_forward    3
turn_right      3
준비              1
연결              1
이륙              1
정지비행            1
착륙              1
Name: count, dtype: int64


value\_counts()는 같은 값이 몇 번 나왔는지 셍니다. 앞으로 가기와 오른쪽 돌기가 각각 3번씩 있었음을 한눈에 알 수 있습니다.

### ====== 해 보기 ======

**구간별 배터리 소모량 구하기**

diff()는 바로 앞 행과의 차이를 구해 줍니다. 이를 이용해 어느 동작에서 배터리를 가장 많이 썼는지 찾아봅시다.

1. df["소모량"] = -df["배터리"].diff()로 새 열을 만듭니다. (줄어든 양이므로 부호를 바꿉니다)
2. sort\_values("소모량", ascending=False)로 정렬합니다.
3. 가장 많이 쓴 동작은 무엇이었는지, 그 까닭은 무엇일지 이야기해 봅시다. 힌트: 그 줄의 바로 앞 줄과 비교해 얼마나 멀리, 얼마나 오래 날았는지 보세요.

In [14]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 데이터프레임은 행·열·인덱스로 이루어진 표 모양의 자료구조입니다.
- read\_csv()로 파일을 불러오고 head(), shape, info()로 전체를 살핍니다.
- df["열이름"]으로 열을, df[조건]으로 조건에 맞는 행을 꺼냅니다.
- sort\_values()로 정렬하고 max(), min(), mean(), value\_counts()로 집계합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 2-3 데이터 수집과 전처리

**학습 목표**

- 공공데이터의 출처와 이용 조건을 확인하고 수집할 수 있습니다.
- 결측값과 이상값을 찾아내고 알맞게 처리할 수 있습니다.
- 자료형이 잘못된 열을 바로잡아 분석할 수 있는 형태로 만들 수 있습니다.

<sub>관련 성취기준 [드코1 02-03-01] / [드코1 02-03-02]</sub>

> **생각 열기**  
> 실제로 내려받은 데이터를 열어 보면 빈칸이 있고, 풍속이 999로 적힌 줄도 있습니다. 이런 데이터로 평균을 내면 어떤 일이 벌어질까요?

<sub>교재에서 읽기: ■ 1. 공공데이터 수집하기</sub>

## ■ 2. 전처리가 필요한 까닭

현실의 데이터는 깨끗하지 않습니다. 센서가 잠시 멈춰 값이 비거나, 고장으로 말이 안 되는 값이 기록되기도 합니다. 단위가 글자와 붙어 있어 숫자로 계산되지 않는 열도 있습니다. 이런 자료를 그대로 분석하면 잘못된 결론에 이릅니다. 분석에 앞서 데이터를 정리하는 일을 전처리라고 합니다.

**예제 2-11. 문제가 있는 기상 데이터 만들기**

In [15]:
# 예제 2-11. 문제가 있는 기상 데이터 만들기
import pandas as pd

data = {
    "날짜": ["10-01", "10-02", "10-03", "10-04", "10-05", "10-06"],
    "풍속": [2.1, 3.4, None, 999.0, 4.2, 3.0],          # None=빠진 값, 999=이상값
    "강수량": [0.0, 0.0, 1.5, 0.0, None, 0.0],
    "기온": ["18℃", "19℃", "17℃", "20℃", "21℃", "18℃"],   # 단위가 붙어 글자가 됨
}
df = pd.DataFrame(data)
df.to_csv("weather.csv", index=False, encoding="utf-8-sig")
print(df)

      날짜     풍속  강수량   기온
0  10-01    2.1  0.0  18℃
1  10-02    3.4  0.0  19℃
2  10-03    NaN  1.5  17℃
3  10-04  999.0  0.0  20℃
4  10-05    4.2  NaN  21℃
5  10-06    3.0  0.0  18℃


비어 있는 값은 NaN(Not a Number, 숫자가 아님)으로 표시됩니다. 10월 4일의 풍속 999는 초속 999 m라는 뜻이 되므로 현실에서는 있을 수 없는 이상값입니다. 실제 관측 자료에서도 센서가 값을 재지 못했을 때 999 같은 특수한 수를 넣어 두는 경우가 있습니다.

## ■ 3. 문제 찾아내기

**예제 2-12. 빠진 값과 요약 통계 확인하기**

In [16]:
# 예제 2-12. 빠진 값과 요약 통계 확인하기
import pandas as pd

df = pd.read_csv("weather.csv")
print("--- 열별 빠진 값의 수 ---")
print(df.isna().sum())
print()
print("--- 풍속 열의 요약 ---")
print(df["풍속"].describe())

--- 열별 빠진 값의 수 ---
날짜     0
풍속     1
강수량    1
기온     0
dtype: int64

--- 풍속 열의 요약 ---
count      5.000000
mean     202.340000
std      445.347121
min        2.100000
25%        3.000000
50%        3.400000
75%        4.200000
max      999.000000
Name: 풍속, dtype: float64


> **짚고 가기 · 평균 202 m/s가 말해 주는 것**  
> 풍속의 평균이 202.34로 나왔습니다. 우리나라에 큰 피해를 남긴 태풍 매미(2003)의 최대순간풍속도 초속 60 m 정도였는데 터무니없는 값입니다. 이상값 999 하나가 평균을 완전히 망가뜨린 것입니다. 반면 가운랫값(50%)은 3.4로 정상에 가깝습니다. 평균은 이상값에 약하다는 점을 기억하고, 분석 전에 describe()로 최솟값과 최댓값을 먼저 확인하는 습관을 들입니다.

## ■ 4. 이상값 처리하기

이상값은 기준을 정해 걸러냅니다. 여기서는 풍속이 30 m/s를 넘는 값을 기록 오류로 보겠습니다. 기상청은 풍속 14 m/s(순간풍속 20 m/s) 이상이 예상될 때 강풍주의보를 내립니다. 하루 평균 풍속이 30 m/s를 넘는 날은 매우 드물게 강한 태풍이 아니면 나타나기 어렵습니다. 걸러낸 값은 지우거나, 빠진 값으로 바꾼 뒤 함께 처리합니다.

**예제 2-13. 이상값을 결측값으로 바꾸기**

In [17]:
# 예제 2-13. 이상값을 결측값으로 바꾸기
import pandas as pd

df = pd.read_csv("weather.csv")
df.loc[df["풍속"] > 30, "풍속"] = None     # 30 초과는 기록 오류로 보고 비움
print(df[["날짜", "풍속"]])
print("\n이상값 처리 후 평균 풍속:", round(df["풍속"].mean(), 2), "m/s")

      날짜   풍속
0  10-01  2.1
1  10-02  3.4
2  10-03  NaN
3  10-04  NaN
4  10-05  4.2
5  10-06  3.0

이상값 처리 후 평균 풍속: 3.18 m/s


평균이 202.34에서 3.18로 바뀌었습니다. 같은 데이터라도 전처리를 했는지에 따라 결론이 완전히 달라집니다. df.loc[조건, "열"]은 '조건에 맞는 줄의 그 열'을 가리킵니다. 거기에 None을 넣어 빈칸으로 만든 것입니다.

## ■ 5. 결측값 처리하기

빠진 값을 다루는 방법은 크게 두 가지입니다. 어느 쪽이 좋은지는 상황에 따라 다릅니다.

| 방법 | 명령 | 쓰기 좋은 경우 | 주의할 점 |
|---|---|---|---|
| 삭제하기 | dropna() | 데이터가 충분히 많을 때 | 자료가 줄어듭니다 |
| 채우기 | fillna(값) | 자료가 적어 버리기 아까울 때 | 실제로 잰 값이 아닙니다 |

<sub>표 2-4. 결측값을 다루는 두 가지 방법</sub>

**예제 2-14. 두 가지 방법 비교하기**

In [18]:
# 예제 2-14. 두 가지 방법 비교하기
import pandas as pd

df = pd.read_csv("weather.csv")
df.loc[df["풍속"] > 30, "풍속"] = None

drop_df = df.dropna()                    # 방법 1: 빠진 줄 삭제
print("삭제 후 남은 줄 수:", len(drop_df))

fill_df = df.copy()                      # 방법 2: 채우기
fill_df["풍속"] = fill_df["풍속"].fillna(round(df["풍속"].mean(), 2))
fill_df["강수량"] = fill_df["강수량"].fillna(0)
print("채운 후 남은 줄 수:", len(fill_df))
print()
print(fill_df[["날짜", "풍속", "강수량"]])

삭제 후 남은 줄 수: 3
채운 후 남은 줄 수: 6

      날짜    풍속  강수량
0  10-01  2.10  0.0
1  10-02  3.40  0.0
2  10-03  3.18  1.5
3  10-04  3.18  0.0
4  10-05  4.20  0.0
5  10-06  3.00  0.0


여섯 줄짜리 작은 데이터에서 삭제를 고르면 절반이 사라집니다. 이럴 때는 채우는 쪽이 나을 수 있습니다. 다만 채워 넣은 값은 실제로 잰 값이 아니므로, 분석 결과를 보고할 때 반드시 그 사실을 밝힙니다. 강수량을 0으로 채운 것도 판단입니다. '기록이 없으면 비가 오지 않았다'고 본 것인데, 실제로는 비가 왔는데 기록이 빠졌을 수도 있습니다.

## ■ 6. 자료형 바로잡기

기온 열은 18℃처럼 단위가 붙어 있어서 글자로 저장되었습니다. 이 상태로는 평균을 구할 수 없습니다. astype(float)로 바로 바꾸려 하면 ValueError: could not convert string to float: '18℃' 오류가 납니다. 문자 ℃를 먼저 지우고 바꿔야 합니다.

**예제 2-15. 단위를 지우고 숫자로 바꾸기**

In [19]:
# 예제 2-15. 단위를 지우고 숫자로 바꾸기
import pandas as pd

df = pd.read_csv("weather.csv")
print("바꾸기 전:", df["기온"].dtype)

df["기온"] = df["기온"].str.replace("℃", "").astype(float)
print("바꾼 후  :", df["기온"].dtype)
print("평균 기온:", round(df["기온"].mean(), 1), "도")

바꾸기 전: str
바꾼 후  : float64
평균 기온: 18.8 도


.str은 '이 열의 값을 문자열로 다루겠다'는 표시이고, replace("℃", "")는 ℃를 빈 글자로 바꿉니다. pandas 버전에 따라 '바꾸기 전'이 object로 표시되기도 합니다. 둘 다 글자라는 뜻입니다.

> **한 걸음 더 · 전처리 기록을 남깁니다.**  
> 어떤 값을 왜 지우거나 바꾸었는지 기록으로 남겨야 합니다. 같은 자료로 다른 사람이 분석했을 때 결과가 다르면, 전처리 과정이 달랐기 때문인 경우가 많습니다. 코드에 주석으로 '# 풍속 30 초과는 기록 오류로 판단하여 제외'처럼 판단의 근거를 함께 적어 둡니다.

### ====== 해 보기 ======

**우리 지역 날씨로 해 보기**

1. 기상자료개방포털에서 우리 지역 관측소의 종관기상관측(ASOS) 일자료를 최근 한 달치 내려받습니다. 평균풍속, 최대풍속, 최대순간풍속 항목을 고릅니다.
2. 코랩에 올리고 pd.read\_csv("파일이름.csv", encoding="cp949")로 불러옵니다. 이 파일은 한글 윈도우 방식(cp949)으로 저장되어 있어 encoding을 빼면 한글이 깨지거나 오류가 납니다.
3. 자료의 출처, 작성 기관, 이용 조건을 표로 정리합니다.
4. isna().sum()과 describe()로 빠진 값과 이상값이 있는지 확인합니다.
5. 처리 방법을 정하고, 그렇게 정한 까닭을 한 문장으로 적습니다.
6. 전처리 전후의 평균 풍속을 비교합니다.

In [20]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 공공데이터는 출처, 이용 조건, 내용을 확인한 뒤 수집하며 개인정보가 담긴 자료는 쓰지 않습니다.
- 결측값은 NaN으로 표시되며 isna().sum()으로 개수를 확인합니다.
- 이상값은 기준을 정해 걸러내고, describe()로 최솟값·최댓값을 먼저 살핍니다.
- 결측값은 삭제하거나 채울 수 있으며, 어떤 처리를 했는지 반드시 기록으로 남깁니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 2-4 기상 및 비행 데이터 분석

**학습 목표**

- 기상 요소가 드론 비행에 미치는 영향을 설명할 수 있습니다.
- 기상 데이터를 분석해 비행할 수 있는 날을 가려낼 수 있습니다.
- 분석 결과를 근거로 운항 계획을 세우고 설명할 수 있습니다.

<sub>관련 성취기준 [드코1 02-04-01] / [드코1 02-04-02]</sub>

> **생각 열기**  
> '내일 배송 비행을 해도 될까?' 이 질문에 '아마 괜찮을 것 같다'가 아니라 '풍속이 기준보다 낮고 비가 오지 않으므로 가능하다'라고 답하려면 무엇이 필요할까요?

<sub>교재에서 읽기: ■ 1. 기상이 비행에 미치는 영향</sub>

## ■ 2. 비행할 수 있는 날 가려내기

기준을 정하고 그 기준으로 데이터를 걸러내면 비행할 수 있는 날을 가려낼 수 있습니다. 여기서는 풍속 5 m/s 이하, 강수량 0 mm를 기준으로 삼습니다. 아래 날씨 값은 연습을 위해 만든 것입니다.

**예제 2-16. 비행 가능일 찾기**

In [21]:
# 예제 2-16. 비행 가능일 찾기
import pandas as pd

data = {
    "날짜": ["10-01", "10-02", "10-03", "10-04", "10-05", "10-06", "10-07", "10-08"],
    "풍속": [2.1, 3.4, 6.8, 4.2, 7.5, 3.0, 1.8, 5.9],
    "강수량": [0.0, 0.0, 1.5, 0.0, 12.0, 0.0, 0.0, 0.0],
    "기온": [18.0, 19.0, 17.0, 20.0, 15.0, 18.0, 21.0, 16.0],
}
df = pd.DataFrame(data)

WIND_LIMIT = 5.0
df["비행가능"] = (df["풍속"] <= WIND_LIMIT) & (df["강수량"] == 0)
print(df)

ok = df[df["비행가능"]]
print("\n비행 가능한 날:", len(ok), "일 /", len(df), "일")
print("해당 날짜:", list(ok["날짜"]))

      날짜   풍속   강수량    기온   비행가능
0  10-01  2.1   0.0  18.0   True
1  10-02  3.4   0.0  19.0   True
2  10-03  6.8   1.5  17.0  False
3  10-04  4.2   0.0  20.0   True
4  10-05  7.5  12.0  15.0  False
5  10-06  3.0   0.0  18.0   True
6  10-07  1.8   0.0  21.0   True
7  10-08  5.9   0.0  16.0  False

비행 가능한 날: 5 일 / 8 일
해당 날짜: ['10-01', '10-02', '10-04', '10-06', '10-07']


여덟 날 가운데 다섯 날에 비행할 수 있다는 결론을 얻었습니다. 중요한 것은 이 결론이 '풍속 5 m/s 이하, 강수량 0 mm'라는 기준에 기대고 있다는 점입니다. 기준을 바꾸면 결과도 달라지므로, 결과를 말할 때는 기준도 함께 밝힙니다. WIND\_LIMIT처럼 대문자로 쓴 변수는 '프로그램이 도는 동안 바꾸지 않을 기준값'이라는 표시입니다.

> **안전 수칙 · 분석 결과가 안전을 보장하지는 않습니다.**  
> 데이터 분석으로 '비행 가능'이라는 결과가 나왔더라도, 실제 비행 여부는 현장에서 다시 확인해야 합니다. 예보와 실제 날씨는 다를 수 있고, 건물 사이의 돌풍처럼 데이터에 나타나지 않는 위험도 있습니다. 비행 제한 구역과 관련 법규는 반드시 공식 자료로 확인합니다.

## ■ 3. 가상 드론으로 속도별 효율 실험하기

같은 거리를 날더라도 속도에 따라 비행 시간과 배터리 소모가 달라집니다. 느리게 가면 배터리를 아낄까요, 빨리 가면 아낄까요? 예상을 적어 둔 뒤 가상 드론으로 3 m를 세 가지 속도로 날아 보고, 그 기록을 데이터프레임으로 만들어 비교합니다.

**예제 2-17. 속도에 따른 효율 비교**

In [22]:
# 예제 2-17. 속도에 따른 효율 비교
from drone_sim import *
import pandas as pd

rows = []
for speed in [0.3, 0.5, 0.8]:                 # m/s
    drone = Drone(verbose=False)              # 가상 전용: 안내문 끄기
    drone.pair()
    drone.takeoff()
    start_t = drone.sim_log[-1]["시간"]        # 가상 전용: 기록 꺼내기
    start_b = drone.sim_log[-1]["배터리"]
    drone.move_forward(300, speed=speed)       # 3 m 비행
    end = drone.sim_log[-1]
    rows.append({"속도": speed,
                 "비행시간": round(end["시간"] - start_t, 2),
                 "배터리소모": round(start_b - end["배터리"], 2)})
    drone.land()
    drone.close()

df = pd.DataFrame(rows)
df["효율"] = round(300 / df["배터리소모"], 1)   # 배터리 1 %로 가는 거리(cm)
print(df)

best = df.sort_values("효율", ascending=False).iloc[0]
print("\n효율이 가장 좋은 속도:", best["속도"], "m/s")

    속도   비행시간  배터리소모     효율
0  0.3  10.00    2.3  130.4
1  0.5   6.00    1.5  200.0
2  0.8   3.75    1.1  272.7

효율이 가장 좋은 속도: 0.8 m/s


느리게 가면 배터리를 아낄 거라고 예상했다면 결과가 반대입니다. 드론은 제자리에 떠 있는 것만으로도 배터리를 씁니다. 느리게 가면 그만큼 오래 떠 있어야 하므로, 같은 3 m를 가는 데 오히려 더 많이 쓰는 것입니다. 자동차가 걸어가는 속도로 달리면 연비가 나쁜 것과 비슷합니다.

> **짚고 가기 · 이 결론은 어디까지 믿을 수 있을까요?**  
> 이 표는 가상 드론의 계산식에서 나온 값입니다. 가상 드론은 '떠 있는 데 드는 힘 + 빠르게 움직이는 데 드는 힘'으로 배터리를 줄이도록 만들어져 있습니다. 실제 드론은 너무 빠르면 공기 저항이 커져 다시 효율이 떨어지므로, 가장 좋은 속도는 그 사이 어딘가에 있습니다. 모형에서 얻은 결론은 실기체로 다시 확인해야 할 가설입니다.

> **한 걸음 더 · 실제 배송 드론으로 확인해 보니**  
> 미국 카네기멜런대학교 연구팀은 배송용 드론(DJI Matrice 100)으로 같은 삼각형 경로를 날면서 속도·화물 무게·높이를 정해진 값으로 바꿔 가며 180여 번 비행하고, 비행마다 쓴 에너지를 쟀습니다(부록 5). 속도별로 평균을 내면 비행 1회에 4 m/s는 27.5 Wh, 6 m/s는 22.1 Wh, 8 m/s는 19.9 Wh, 10 m/s는 18.4 Wh, 12 m/s는 18.5 Wh를 썼습니다.  
> 4 m/s에서 10 m/s까지는 빠를수록 에너지를 덜 썼습니다. 평균 전력은 속도와 관계없이 400 W 안팎으로 거의 같았고, 빨리 날아 비행 시간이 짧아진 것이 그 까닭입니다. 예제 2-17의 가상 드론과 같은 원리입니다. 그러나 12 m/s는 10 m/s보다 나아지지 않았습니다. 그래서 가설은 '맞았다'가 아니라 '이 기체, 이 경로에서는 10 m/s 근처까지 맞았다'로 정리해야 합니다. 이 기체는 코드론 EDU와 크기도 속도 범위도 전혀 다르므로 숫자를 그대로 옮길 수 없습니다. 연계 실습 노트 파트 3에서 이 자료를 직접 분석해 봅니다.

효율만이 답은 아닙니다. '무엇을 중요하게 볼 것인가'에 따라 답이 달라집니다. 빠르게 날수록 바람이나 장애물에 대처할 시간이 짧아지고, 실내에서는 부딪혔을 때 충격도 커집니다.

## ■ 4. 운항 계획 세우기

분석 결과를 모아 운항 계획을 세워 봅시다. 좋은 계획에는 결론과 함께 그 근거가 들어 있습니다. 1초당 배터리 소모량은 1-3의 예제 1-6과 같은 방법으로 구합니다.

**예제 2-18. 분석 결과로 운항 계획 요약하기**

In [23]:
# 예제 2-18. 분석 결과로 운항 계획 요약하기
import pandas as pd

weather = pd.DataFrame({
    "날짜": ["10-01", "10-02", "10-03", "10-04"],
    "풍속": [2.1, 3.4, 6.8, 4.2],
    "강수량": [0.0, 0.0, 1.5, 0.0],
})
WIND_LIMIT, SPEED = 5.0, 50
USE_PER_SEC = 100 / (7.5 * 60)       # 1-3 예제 1-6과 같은 가정
DISTANCE = 1200                      # 왕복 비행 거리(cm)

weather["비행가능"] = (weather["풍속"] <= WIND_LIMIT) & (weather["강수량"] == 0)
need_time = DISTANCE / SPEED
need_battery = round(need_time * USE_PER_SEC, 1)
days = list(weather[weather["비행가능"]]["날짜"])

print("[운항 계획 요약]")
print(f" 기준       : 풍속 {WIND_LIMIT} m/s 이하, 강수량 0 mm")
print(f" 왕복 거리  : {DISTANCE} cm, 속도 {SPEED} cm/s")
print(f" 소요 시간  : {need_time} 초")
print(f" 필요 배터리: {need_battery} % (여유분 20 % 별도)")
print(f" 비행 가능일: {days}")

[운항 계획 요약]
 기준       : 풍속 5.0 m/s 이하, 강수량 0 mm
 왕복 거리  : 1200 cm, 속도 50 cm/s
 소요 시간  : 24.0 초
 필요 배터리: 5.3 % (여유분 20 % 별도)
 비행 가능일: ['10-01', '10-02', '10-04']


### ====== 해 보기 ======

**우리 모둠 운항 계획 발표하기**

2-3에서 모은 우리 지역 기상 자료로 운항 계획을 세우고 발표해 봅시다.

1. 비행 기준(풍속·강수량)을 모둠에서 정하고, 왜 그 값으로 정했는지 적습니다. 평균풍속과 최대순간풍속 가운데 무엇을 기준으로 삼을지도 정합니다.
2. 기준에 맞는 날을 골라내고, 전체 날 수에 대한 비율을 구합니다.
3. 비행 거리와 속도를 정해 필요한 배터리를 계산합니다.
4. '결론 한 문장 + 근거 세 가지' 형식으로 정리해 발표합니다.
5. 다른 모둠과 기준이 달랐다면 왜 달랐는지 이야기해 봅시다.

In [24]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 풍속·강수·기온·시정은 드론 비행에 직접 영향을 주며, 특히 바람이 중요합니다.
- 기준을 정해 데이터를 걸러내면 비행할 수 있는 날을 가려낼 수 있습니다.
- 함께 변하는 두 값이 보여도 곧바로 원인과 결과로 단정하지 않고, 모형에서 얻은 결론은 가설로 다룹니다.
- 분석 결과는 운항 계획의 근거가 되지만, 현장 확인과 법규 확인을 대신하지는 못합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# Ⅱ단원 마무리

### 배운 내용 한눈에 보기

| 소단원 | 꼭 기억할 것 |
|---|---|
| 2-1 표준·외부 라이브러리 활용 | import로 라이브러리를 불러오고, 사용법은 지금 쓰는 판의 공식 문서에서 확인합니다. |
| 2-2 데이터프레임 활용 | 데이터프레임은 행·열·인덱스로 된 표입니다. 고르고, 거르고, 정렬하고, 집계해 비행 데이터의 특징을 찾습니다. |
| 2-3 데이터 수집과 전처리 | 공공데이터는 출처와 이용 조건을 확인해 수집합니다. 결측값·이상값·자료형을 바로잡고 어떤 처리를 했는지 기록합니다. |
| 2-4 기상 및 비행 데이터 분석 | 기준을 밝히고 비행 가능일을 가려냅니다. 함께 변하는 두 값을 곧바로 원인과 결과로 단정하지 않고, 모형의 결론은 가설로 다룹니다. |

### 스스로 점검하기

단원을 시작할 때 세운 목표를 얼마나 이루었는지 표시해 봅니다. ○는 '혼자 할 수 있다', △는 '책을 보면 할 수 있다', ✕는 '아직 어렵다'입니다.

| 점검 내용 | ○ △ ✕ |
|---|---|
| 표준·외부 라이브러리를 설치하고 불러와 활용할 수 있다. |  |
| 데이터프레임으로 비행 데이터를 불러와 선택·정렬·집계할 수 있다. |  |
| 공공데이터를 수집하고 결측값과 이상값을 처리할 수 있다. |  |
| 기상·비행 데이터를 분석해 운항 계획의 근거를 제시할 수 있다. |  |

△나 ✕에 표시한 항목은 관련 소단원의 핵심 정리와 예제로 돌아가 다시 살펴봅니다.